# 03 — Uncertainty + Calibration (MC-Dropout, ECE) on KAGGLE

Adds deployment-trust signals to the winning U-Net:
- **MC-Dropout**: dropout kept active at inference, ~30 stochastic passes → per-pixel uncertainty heatmap.
- **ECE + reliability diagram**: is the model's confidence honest?

**Scientific honesty:** this uses a *dropout-enabled U-Net variant*, trained here. The headline segmentation
number (78.5 IoU) stays the original 3-seed U-Net; this variant is purpose-built for uncertainty and reported
as such. Use **GPU T4** (not P100 — P100 hits a CUDA sm_60 mismatch).

In [ ]:
import os
os.chdir('/kaggle/working')
if not os.path.isdir('SteelDefectX'):
    !git clone https://github.com/d-mondal/SteelDefectX.git
os.chdir('/kaggle/working/SteelDefectX')
!git pull
!pip install -q segmentation-models-pytorch albumentations torchmetrics

DATA_ROOT = '/kaggle/working/sdx_data'
if not os.path.isdir(f'{DATA_ROOT}/train'):
    !apt-get -qq install git-lfs
    !git lfs install
    !git clone https://huggingface.co/datasets/Zhaosxian/SteelDefectX {DATA_ROOT}

os.makedirs('data/splits', exist_ok=True)
!python -m src.data.make_split --train-text {DATA_ROOT}/train-text.json --out-dir data/splits --val-frac 0.15 --seed 42

import torch
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## 1. Train ONE dropout-enabled U-Net (~1h on T4)

In [ ]:
import torch, dataclasses
from torch.utils.data import DataLoader
from src.config import Config
from src.data.dataset import make_datasets
from src.segmentation.model import build_model
from src.segmentation.losses import DiceBCELoss
from src.segmentation.metrics import SegMetricAccumulator
from src.segmentation.train import set_seed, evaluate

RESULTS_DIR = '/kaggle/working/results/uncertainty'; os.makedirs(RESULTS_DIR, exist_ok=True)
CKPT = '/kaggle/working/models/unet_dropout.pt'
os.makedirs('/kaggle/working/models', exist_ok=True)

cfg = Config(data_root=DATA_ROOT, split_dir='data/splits', encoder='resnet34',
             batch_size=16, epochs=40, out_dir=RESULTS_DIR, ckpt_dir='/kaggle/working/models')
device = 'cuda'
set_seed(0)

train_ds, val_ds, test_ds = make_datasets(cfg)
tl = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=2, pin_memory=True)
vl = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=2, pin_memory=True)
xl = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=2, pin_memory=True)

model = build_model('unet', cfg.encoder, cfg.encoder_weights, dropout=0.2).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max', factor=0.5, patience=3)
crit = DiceBCELoss(dice_weight=cfg.dice_weight)

best, best_state, noimp = -1, None, 0
from tqdm import tqdm
for ep in range(cfg.epochs):
    model.train(); run=0
    for img, msk, _ in tqdm(tl, desc=f'e{ep}', leave=False):
        img, msk = img.to(device), msk.to(device)
        opt.zero_grad(); loss = crit(model(img), msk); loss.backward(); opt.step()
        run += loss.item()*img.size(0)
    vm = evaluate(model, vl, device, cfg); vi = vm['IoU']; sched.step(vi)
    print(f'e{ep:02d} loss={run/len(train_ds):.4f} valIoU={vi:.2f}')
    if vi>best: best, best_state, noimp = vi, {k:v.cpu().clone() for k,v in model.state_dict().items()}, 0
    else:
        noimp+=1
        if noimp>=cfg.early_stop_patience: print(f'early stop e{ep}'); break
model.load_state_dict(best_state); torch.save(best_state, CKPT)
test_m = evaluate(model, xl, device, cfg)
print('dropout-U-Net test IoU:', test_m['IoU'], '(headline U-Net was 78.5; this variant may differ slightly)')

## 2. Calibration: ECE + reliability diagram (deterministic pass)

In [ ]:
import json
from src.segmentation.uncertainty import evaluate_calibration

cal = evaluate_calibration(model, xl, device=device, n_bins=15, mc_passes=0)
print('ECE (deterministic):', cal['ECE'], '%')
json.dump(cal, open(f'{RESULTS_DIR}/calibration.json','w'), indent=2)

import matplotlib.pyplot as plt, numpy as np
bins = [b for b in cal['bins'] if b['weight']>0]
conf = [b['conf'] for b in bins]; acc = [b['acc'] for b in bins]
plt.figure(figsize=(5,5))
plt.plot([0,1],[0,1],'k--',label='perfect')
plt.plot(conf, acc, 'o-', label=f"model (ECE={cal['ECE']}%)")
plt.xlabel('confidence'); plt.ylabel('accuracy'); plt.title('Reliability diagram'); plt.legend(); plt.show()

## 3. MC-Dropout: per-pixel uncertainty on example images

In [ ]:
from src.segmentation.uncertainty import mc_dropout_predict

# grab a batch of test images spanning a few classes
imgs, masks, classes = next(iter(xl))
mean_p, unc = mc_dropout_predict(model, imgs[:6], n_passes=30, device=device)

fig, ax = plt.subplots(6, 4, figsize=(13, 18))
for r in range(6):
    im = imgs[r,0].numpy(); gt = masks[r,0].numpy()
    pr = mean_p[r,0].numpy(); un = unc[r,0].numpy()
    ax[r,0].imshow(im, cmap='gray'); ax[r,0].set_title(f'{classes[r]}')
    ax[r,1].imshow(gt, cmap='gray'); ax[r,1].set_title('ground truth')
    ax[r,2].imshow(pr, cmap='viridis', vmin=0, vmax=1); ax[r,2].set_title('mean prob')
    im3 = ax[r,3].imshow(un, cmap='inferno'); ax[r,3].set_title('uncertainty (std)')
    for c in range(4): ax[r,c].axis('off')
plt.tight_layout(); plt.savefig(f'{RESULTS_DIR}/uncertainty_examples.png', dpi=110); plt.show()
print('high uncertainty should concentrate on defect BOUNDARIES and ambiguous/diffuse regions')

## 4. MC-Dropout mean uncertainty over the whole test set

In [ ]:
cal_mc = evaluate_calibration(model, xl, device=device, n_bins=15, mc_passes=30)
print('ECE (MC-Dropout mean):', cal_mc['ECE'], '% | mean per-pixel uncertainty:', cal_mc.get('mean_uncertainty'))
json.dump(cal_mc, open(f'{RESULTS_DIR}/calibration_mc.json','w'), indent=2)
print('\nDownload from the Output tab: results/uncertainty/*.json + uncertainty_examples.png')